# bt_laura - Mapping & Evaluation

This notebook runs mapping and evaluation for the **bt_laura** QA results.

It reuses `mapping.py` and evaluation scripts from `prompt-ablation/`.

## Setup

In [ ]:
import sys
import os
import subprocess

# --- Detect environment ---
if 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ:
    PROJECT_ROOT = '/content/drive/MyDrive/AskQE_DNLP_2025-2026'
    print('Running on Colab')
elif 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    PROJECT_ROOT = '/kaggle/working/AskQE_DNLP_2025-2026'
    print('Running on Kaggle')
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
    print(f'Running locally, PROJECT_ROOT: {PROJECT_ROOT}')

In [ ]:
import json

BT_LAURA_DIR = f'{PROJECT_ROOT}/results Qwen3B baseline/biomqm/bt_laura'
ABLATION_DIR = f'{PROJECT_ROOT}/results Qwen3B baseline/biomqm/prompt-ablation'
QG_PATH = f'{BT_LAURA_DIR}/QG/qwen-3b.jsonl'
ORIGINAL_DATASET = f'{PROJECT_ROOT}/biomqm/dev_with_backtranslation.jsonl'

STRATEGY = 'vanilla'
LANGUAGES = ['de', 'es', 'fr', 'ru', 'zh-CN']

# Add prompt-ablation dir to path for imports (mapping.py, evaluation scripts)
if ABLATION_DIR not in sys.path:
    sys.path.insert(0, ABLATION_DIR)

print(f'BT_LAURA_DIR: {BT_LAURA_DIR}')
print(f'QG_PATH: {QG_PATH}')
print(f'ORIGINAL_DATASET: {ORIGINAL_DATASET}')

## Verify QA Files Exist

In [ ]:
print('=== Checking QA files ===')
all_ok = True

# Source
src_file = f'{BT_LAURA_DIR}/QA/source-{STRATEGY}.jsonl'
exists = os.path.exists(src_file)
print(f"  {'✓' if exists else '✗'} source-{STRATEGY}.jsonl")
if not exists: all_ok = False

# BT languages
for lang in LANGUAGES:
    bt_file = f'{BT_LAURA_DIR}/QA/bt-{lang}-{STRATEGY}.jsonl'
    exists = os.path.exists(bt_file)
    print(f"  {'✓' if exists else '✗'} bt-{lang}-{STRATEGY}.jsonl")
    if not exists: all_ok = False

if all_ok:
    print('\n=== All QA files found! ===')
else:
    print('\n=== WARNING: Some files missing! ===')

## Run Mapping

In [ ]:
from mapping import run_mapping

print(f'=== Mapping {STRATEGY} ===')

# QA files are directly in bt_laura/QA/ (flat structure)
qa_dir = f'{BT_LAURA_DIR}/QA'
output_dir = f'{BT_LAURA_DIR}/QA/mapped'

rows = run_mapping(
    strategy=STRATEGY,
    qg_path=QG_PATH,
    original_dataset_path=ORIGINAL_DATASET,
    qa_dir=qa_dir,
    output_dir=output_dir
)
print(f'Completed: {rows} rows mapped')

## Run SBERT Evaluation

In [ ]:
print(f'=== SBERT Evaluation ===')
mapping_file = f'{BT_LAURA_DIR}/QA/mapped/all-{STRATEGY}.jsonl'
sbert_output = f'{BT_LAURA_DIR}/QA/mapped/evaluation/sbert'

sbert_script = f'{ABLATION_DIR}/evaluation/sbert.py'

cmd = [
    sys.executable, '-u',
    sbert_script,
    '--input_path', mapping_file,
    '--output_dir', sbert_output
]
subprocess.run(cmd, check=True)
print('SBERT done!')

## Run String Comparison Evaluation

In [ ]:
print(f'=== String Comparison Evaluation ===')
mapping_file = f'{BT_LAURA_DIR}/QA/mapped/all-{STRATEGY}.jsonl'
sc_output = f'{BT_LAURA_DIR}/QA/mapped/evaluation/string-comparison'

sc_script = f'{ABLATION_DIR}/evaluation/string_comparison.py'

cmd = [
    sys.executable, '-u',
    sc_script,
    '--input_path', mapping_file,
    '--output_dir', sc_output
]
subprocess.run(cmd, check=True)
print('String Comparison done!')

## Results Summary

In [ ]:
import pandas as pd

print('\n' + '='*60)
print('bt_laura RESULTS')
print('='*60)

eval_dir = f'{BT_LAURA_DIR}/QA/mapped/evaluation'

# SBERT
sbert_file = f'{eval_dir}/sbert_summary_by_lang.csv'
if os.path.exists(sbert_file):
    sbert_df = pd.read_csv(sbert_file)
    print('\n--- SBERT (by language) ---')
    print(sbert_df.to_string(index=False))
    print(f'\nAvg SBERT similarity: {sbert_df["avg_similarity"].mean():.4f}')
else:
    print('SBERT results not found')

# String Comparison
sc_file = f'{eval_dir}/string_comparison_summary_by_lang.csv'
if os.path.exists(sc_file):
    sc_df = pd.read_csv(sc_file)
    print('\n--- String Comparison (by language) ---')
    print(sc_df.to_string(index=False))
    print(f'\nAvg F1: {sc_df["avg_f1"].mean():.4f}')
    print(f'Avg EM: {sc_df["avg_em"].mean():.4f}')
    if 'avg_bleu' in sc_df.columns:
        print(f'Avg BLEU: {sc_df["avg_bleu"].mean():.4f}')
    if 'avg_chrf' in sc_df.columns:
        print(f'Avg chrF: {sc_df["avg_chrf"].mean():.4f}')
else:
    print('String Comparison results not found')

## Summary

In [ ]:
print('\n' + '='*60)
print('bt_laura MAPPING & EVALUATION COMPLETE')
print('='*60)
print(f'\nResults saved to: {BT_LAURA_DIR}')
print(f'  - QA/mapped/all-{STRATEGY}.jsonl')
print(f'  - QA/mapped/evaluation/sbert/')
print(f'  - QA/mapped/evaluation/string-comparison/')